In [1]:
import sys
subprocess_result = __import__('subprocess').check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "langchain", "langchain-community", "langchain-core",
    "langchain-text-splitters", "langchain-groq", 
    "faiss-cpu", "sentence-transformers", "pypdf"
])

from langchain_groq import ChatGroq
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

In [ ]:
GROQ_API_KEY = "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"


llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.5,
    api_key=GROQ_API_KEY
)


In [18]:
pdf_path = r"D:\D\work\code\file-sample_150kB.pdf"     # ← Yeh path sahi hai na?

loader = PyPDFLoader(pdf_path)
documents = loader.load()

In [20]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap=50)
chunks = text_splitter.split_documents(documents)


embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [22]:
prompt = ChatPromptTemplate.from_template("""
Answer the question based on the given context only.
If the answer is not in the context, say "I don't know".

Context: {context}

Question: {question}
Answer:
""")

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("✅ RAG System Ready! Ab sawal poochho\n")

✅ RAG System Ready! Ab sawal poochho



In [23]:
while True:
    user_input = input("You: ")
    if user_input.lower() in ['exit', 'quit', 'bye']:
        print("👋 Goodbye! Kal milte hain.")
        break
    
    response = rag_chain.invoke(user_input)
    print(f"Bot: {response}\n")

Bot: The first paragraph describes various design elements and their arrangements. It mentions "Etiam vehicula luctus fermentum" and describes its placement in relation to other elements, such as "vel metus congue" and "fermentum dui". It also talks about "Maecenas ante orci" and its relation to "aliquet" and "sagittis a magna". Additionally, it describes the appearance of "Ut ullamcorper justo sapien" and "Vivamus auctor imperdiet urna".

👋 Goodbye! Kal milte hain.


In [27]:
from langchain_groq import ChatGroq
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

GROQ_API_KEY = "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"

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.5,
    api_key=GROQ_API_KEY
)


pdf_path = r"D:\D\work\code\file-sample_150kB.pdf"  


loader = PyPDFLoader(pdf_path)
documents = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=700, chunk_overlap=50)
chunks = text_splitter.split_documents(documents)

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Answer using only the provided context. If not found, say 'I don't know'."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{question}")
])

def retrieve_and_format(question):
    docs = retriever.invoke(question)
    return format_docs(docs)

base_chain = (
    RunnablePassthrough.assign(context=lambda x: retrieve_and_format(x["question"]))
    | prompt
    | llm
    | StrOutputParser()
)

store = {}

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

conversational_rag = RunnableWithMessageHistory(
    base_chain,
    get_session_history,
    input_messages_key="question",
    history_messages_key="history"
)

print("✅ Conversational RAG Ready!\n")

while True:
    user_input = input("You: ")
    if user_input.lower() in ['exit', 'quit', 'bye']:
        print("👋 Goodbye!")
        break
    
    response = conversational_rag.invoke(
        {"question": user_input},
        config={"configurable": {"session_id": "session_1"}}
    )
    print(f"Bot: {response}\n")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ Conversational RAG Ready!



c:\Users\Ali Mehdi\anaconda3\Lib\site-packages\IPython\core\interactiveshell.py:3699: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


Bot: I don't know.

Bot: Aapne poocha tha "What is this document about?"

Bot: Mujhe khed hai, lekin main is document ke baare mein kuchh nahin jaanta, isliye main summary nahin de sakta.

Bot: Aapne pehle "What is this document about?" poocha tha, phir "Pehle maine kya poocha tha?" aur phir "Summary do is document ka".

👋 Goodbye!
